# A Spam Classifier
The work never ends! We must practice however, we are going to make a spam classifier here, 
hopefully I can create this in less time than the last project with the time we have.

This time though, our teacher has given us instructions. They go as follows:

```python
# 1. Download examples of spam and ham from Apache SpamAssassin’s public datasets.

# 2. Unzip the datasets and familiarize yourself with the data format.

# 3. Split the datasets into a training set and a test set.

# 4. Write a data preparation pipeline to convert each email into a feature vector. Your preparation 
#   pipeline should transform an email into a (sparse) vector that indicates the presence or absence 
#   of each possible word. For example, if all emails only ever contain four words, “Hello,” “how,” 
#   “are,” “you,” then the email “Hello you Hello Hello you” would be converted into a vector [1, 0, 
#   0, 1] (meaning [“Hello” is present, “how” is absent, “are” is absent, “you” is present]), or [3, 
#   0, 0, 2] if you prefer to count the number of occurrences of each word.

# 5. You may want to add hyperparameters to your preparation pipeline to control whether or not to 
#   strip off email headers, convert each email to lowercase, remove punctuation, replace all URLs 
#   with “URL,” replace all numbers with “NUMBER,” or even perform stemming (i.e., trim off word 
#   endings; there are Python libraries available to do this).

# 6. Finally, try out several classifiers and see if you can build a great spam classifier, with 
#   both high recall and high precision
```

In [12]:
# fetch requirements
!pip install -r requirements.txt

In [1]:
# imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

extract_files = False

In [2]:
# get ham_2
'''
easy_ham: 2500 non-spam messages.  These are typically quite easy to
differentiate from spam, since they frequently do not contain any spammish
signatures (like HTML etc). -- unused

easy_ham_2: 1400 non-spam messages.  A more recent addition to the set.
'''

import tarfile

with tarfile.open('./data/20030228_easy_ham_2.tar.bz2', 'r:bz2') as f:
    f.list(verbose=True)
    if extract_files: f.extractall() # catch

drwx--x--x jm/jm          0 2003-02-28 06:04:53 easy_ham_2/ 
-rw-rw-r-- jm/jm       2841 2003-02-28 05:50:22 easy_ham_2/01394.b4dd1cece01b908f040e33493643c4a4 
-rw-rw-r-- jm/jm       5887 2003-02-28 05:50:20 easy_ham_2/00002.5a587ae61666c5aa097c8e866aedcc59 
-rw-rw-r-- jm/jm       6321 2003-02-28 05:50:20 easy_ham_2/00003.19be8acd739ad589cd00d8425bac7115 
-rw-rw-r-- jm/jm      12283 2003-02-28 05:50:20 easy_ham_2/00004.b2ed6c3c62bbdfab7683d60e214d1445 
-rw-rw-r-- jm/jm       4876 2003-02-28 05:50:20 easy_ham_2/00005.07b9d4aa9e6c596440295a5170111392 
-rw-rw-r-- jm/jm      11824 2003-02-28 05:50:20 easy_ham_2/00006.654c4ec7c059531accf388a807064363 
-rw-rw-r-- jm/jm       5978 2003-02-28 05:50:20 easy_ham_2/00007.2e086b13730b68a21ee715db145522b9 
-rw-rw-r-- jm/jm       7738 2003-02-28 05:50:20 easy_ham_2/00008.6b73027e1e56131377941ff1db17ff12 
-rw-rw-r-- jm/jm       6265 2003-02-28 05:50:20 easy_ham_2/00009.13c349859b09264fa131872ed4fb6e4e 
-rw-rw-r-- jm/jm       4528 2003-02-28 05:50:20 

In [3]:
# get spam_2

'''
spam: 500 spam messages, all received from non-spam-trap sources. -- unused.

spam_2: 1397 spam messages.  Again, more recent.
'''

with tarfile.open('./data/20050311_spam_2.tar.bz2', 'r:bz2') as f:
    f.list(verbose=True)
    if extract_files: f.extractall()

drwxrwxr-x jm/jm          0 2005-03-11 18:46:58 spam_2/ 
-rw-rw-r-- jm/jm       4721 2003-02-28 05:58:07 spam_2/00001.317e78fa8ee2f54cd4890fdc09ba8176 
-rw-rw-r-- jm/jm       6165 2003-02-28 05:58:07 spam_2/00002.9438920e9a55591b18e60d1ed37d992b 
-rw-rw-r-- jm/jm       6942 2003-02-28 05:58:07 spam_2/00003.590eff932f8704d8b0fcbe69d023b54d 
-rw-rw-r-- jm/jm       7120 2003-02-28 05:58:07 spam_2/00004.bdcc075fa4beb5157b5dd6cd41d8887b 
-rw-rw-r-- jm/jm       4527 2003-02-28 05:58:07 spam_2/00005.ed0aba4d386c5e62bc737cf3f0ed9589 
-rw------- jm/jm      22348 2003-02-28 05:58:07 spam_2/00006.3ca1f399ccda5d897fecb8c57669a283 
-rw------- jm/jm       9360 2003-02-28 05:58:07 spam_2/00151.6abbf42bc1bfb6c36b749372da0cffae 
-rw-rw-r-- jm/jm      12702 2003-02-28 05:58:07 spam_2/00008.ccf927a6aec028f5472ca7b9db9eee20 
-rw-rw-r-- jm/jm       6565 2003-02-28 05:58:07 spam_2/00009.1e1a8cb4b57532ab38aa23287523659d 
-rw-rw-r-- jm/jm       3883 2003-02-28 05:58:07 spam_2/00010.2558d935f6439cb40d3acb8b856

In [4]:
# get hard_ham
''' 
hard_ham: 250 non-spam messages which are closer in many respects to
typical spam: use of HTML, unusual HTML markup, coloured text,
"spammish-sounding" phrases etc.
'''

with tarfile.open('./data/20030228_hard_ham.tar.bz2', 'r:bz2') as f:
    f.list(verbose=True)
    if extract_files: f.extractall()

drwx--x--x jm/jm          0 2004-12-16 14:38:01 hard_ham/ 
-rw------- jm/jm       8318 2003-02-28 05:56:49 hard_ham/00001.7c7d6921e671bbe18ebb5f893cd9bb35 
-rw-r--r-- jm/jm      15866 2003-02-28 05:56:49 hard_ham/00002.ca96f74042d05c1a1d29ca30467cfcd5 
-rw-rw-r-- jm/jm       1003 2003-02-28 05:56:49 hard_ham/00003.268fd170a3fc73bee2739d8204856a53 
-rw-rw-r-- jm/jm       8614 2003-02-28 05:56:49 hard_ham/00004.68819fc91d34c82433074d7bd3127dcc 
-rw------- jm/jm      31102 2003-02-28 05:56:49 hard_ham/00151.b352916ecff2b0ba1140d6898d789235 
-rw-r--r-- jm/jm       6791 2003-02-28 05:56:49 hard_ham/00006.3409dec8ca4fcf2d6e0582554473b5c9 
-rw-r--r-- jm/jm      10849 2003-02-28 05:56:49 hard_ham/00007.d24e99a602ee7fb442714c0d448cd08e 
-rw-r--r-- jm/jm      23547 2003-02-28 05:56:49 hard_ham/00008.b42457819236bee543bebffb61b91e44 
-rw-r--r-- jm/jm       3533 2003-02-28 05:56:49 hard_ham/00009.ddea79a02a9978cb3dafef3c05ff37a6 
-rw-r--r-- jm/jm      20496 2003-02-28 05:56:49 hard_ham/00010.e82bd

### File Layout
These emails follow a similar structure. Below is a sample truncated email:
```python
'''
From exmh-workers-admin@redhat.com  Wed Aug 21 16:46:03 2002
Return-Path: <exmh-workers-admin@spamassassin.taint.org>
Delivered-To: yyyy@localhost.netnoteinc.com
Received: from localhost (localhost [127.0.0.1])
	by phobos.labs.netnoteinc.com (Postfix) with ESMTP id 7B74843C32
	for <jm@localhost>; Wed, 21 Aug 2002 11:46:03 -0400 (EDT)
{...truncated, more headers}
\n
Date: Wed, 21 Aug 2002 10:47:32 -0500

--==_Exmh_-2080822444P
Content-Type: text/plain; charset=us-ascii

> From:  Chris Garrigues <cwg-exmh@DeepEddy.Com>
> Date:  Wed, 21 Aug 2002 10:40:39 -0500
>
> > From:  Chris Garrigues <cwg-exmh@DeepEddy.Com>
{...truncated, more body}
_______________________________________________
Exmh-workers mailing list
Exmh-workers@redhat.com
https://listman.redhat.com/mailman/listinfo/exmh-workers
'''
```
Most emails follow this pattern of format... as such, we can derive the following structure:
1. **From:** `From <email> <date> <time> <year>`
	* note that there are scenarios where this From field is missing
2. **Headers:** `<header name>: data`
	* note that if a header is multi-line, it will continue with a newline and a tab
3. **Newline:** there is always a single newline empty line seperating the Headers from the Body
4. **Body:** any text


### Line Extraction
Logically speaking, it makes sense to split each line into an entry within a list. And perhaps
create a sublist within each line which splits each word up.

In [5]:
# how to convert into a dataframe, lets look through a specific file to see how it reads.
with open('./data/easy_ham_2/00003.19be8acd739ad589cd00d8425bac7115', 'r') as f:
    for line in f:
        print(line, end='') # no /n needed since present

From exmh-workers-admin@redhat.com  Wed Aug 21 16:18:36 2002
Return-Path: <exmh-workers-admin@spamassassin.taint.org>
Delivered-To: yyyy@localhost.netnoteinc.com
Received: from localhost (localhost [127.0.0.1])
	by phobos.labs.netnoteinc.com (Postfix) with ESMTP id 5134943C34
	for <jm@localhost>; Wed, 21 Aug 2002 11:18:36 -0400 (EDT)
Received: from phobos [127.0.0.1]
	by localhost with IMAP (fetchmail-5.9.0)
	for jm@localhost (single-drop); Wed, 21 Aug 2002 16:18:36 +0100 (IST)
Received: from listman.spamassassin.taint.org (listman.spamassassin.taint.org [66.187.233.211]) by
    dogma.slashnull.org (8.11.6/8.11.6) with ESMTP id g7LFJuZ30902 for
    <jm-exmh@jmason.org>; Wed, 21 Aug 2002 16:19:56 +0100
Received: from listman.spamassassin.taint.org (localhost.localdomain [127.0.0.1]) by
    listman.redhat.com (Postfix) with ESMTP id D96EC3EC31; Wed, 21 Aug 2002
    11:20:01 -0400 (EDT)
Delivered-To: exmh-workers@listman.spamassassin.taint.org
Received: from int-mx1.corp.spamassassin.tain

In [6]:
def extract_lines(lines: str):
    data = {
        'frm': [],
        'headers': [],
        'body': []
    }

    idx = 0
    
    # determine if we have a from line
    if ':' not in lines[0].split()[0]:
        data['frm'].append(lines[0])
        idx += 1
    
    # headers...
    # the header and body section are split up by a single \n line. Therefore, '' entry in our split
    while lines[idx] != '\n':
        data['headers'].append(lines[idx])
        idx += 1

    # body
    while idx < len(lines):
        data['body'].append(lines[idx])
        idx += 1

    return data

email1_data = None
with open('./data/easy_ham_2/00003.19be8acd739ad589cd00d8425bac7115', 'r') as f:
    email1_data = extract_lines(f.readlines())


email1_data

{'frm': ['From exmh-workers-admin@redhat.com  Wed Aug 21 16:18:36 2002\n'],
 'headers': ['Return-Path: <exmh-workers-admin@spamassassin.taint.org>\n',
  'Delivered-To: yyyy@localhost.netnoteinc.com\n',
  'Received: from localhost (localhost [127.0.0.1])\n',
  '\tby phobos.labs.netnoteinc.com (Postfix) with ESMTP id 5134943C34\n',
  '\tfor <jm@localhost>; Wed, 21 Aug 2002 11:18:36 -0400 (EDT)\n',
  'Received: from phobos [127.0.0.1]\n',
  '\tby localhost with IMAP (fetchmail-5.9.0)\n',
  '\tfor jm@localhost (single-drop); Wed, 21 Aug 2002 16:18:36 +0100 (IST)\n',
  'Received: from listman.spamassassin.taint.org (listman.spamassassin.taint.org [66.187.233.211]) by\n',
  '    dogma.slashnull.org (8.11.6/8.11.6) with ESMTP id g7LFJuZ30902 for\n',
  '    <jm-exmh@jmason.org>; Wed, 21 Aug 2002 16:19:56 +0100\n',
  'Received: from listman.spamassassin.taint.org (localhost.localdomain [127.0.0.1]) by\n',
  '    listman.redhat.com (Postfix) with ESMTP id D96EC3EC31; Wed, 21 Aug 2002\n',
  '    

In [7]:
# an email without a from line.
email2_data = None
with open('./data/easy_ham_2/00001.1a31cc283af0060967a233d26548a6ce', 'r') as f:
    email2_data = extract_lines(f.readlines())


email2_data

{'frm': [],
 'headers': ['Return-Path: <exmh-workers-admin@spamassassin.taint.org>\n',
  'Delivered-To: yyyy@localhost.netnoteinc.com\n',
  'Received: from localhost (localhost [127.0.0.1])\n',
  '\tby phobos.labs.netnoteinc.com (Postfix) with ESMTP id 7106643C34\n',
  '\tfor <jm@localhost>; Wed, 21 Aug 2002 08:33:03 -0400 (EDT)\n',
  'Received: from phobos [127.0.0.1]\n',
  '\tby localhost with IMAP (fetchmail-5.9.0)\n',
  '\tfor jm@localhost (single-drop); Wed, 21 Aug 2002 13:33:03 +0100 (IST)\n',
  'Received: from listman.spamassassin.taint.org (listman.spamassassin.taint.org [66.187.233.211]) by\n',
  '    dogma.slashnull.org (8.11.6/8.11.6) with ESMTP id g7LCXvZ24654 for\n',
  '    <jm-exmh@jmason.org>; Wed, 21 Aug 2002 13:33:57 +0100\n',
  'Received: from listman.spamassassin.taint.org (localhost.localdomain [127.0.0.1]) by\n',
  '    listman.redhat.com (Postfix) with ESMTP id F12A13EA25; Wed, 21 Aug 2002\n',
  '    08:34:00 -0400 (EDT)\n',
  'Delivered-To: exmh-workers@listman.s

When considering headers and body, there is a lot more work involved. Lets think about the headers
particularly for now...

In general, we want to perform the following actions on headers:
1. Split all words into entries within a list.
2. Extract which header a line(s) belong to.
3. Strip all newlines, tabs, commas, chevrons, and semicolons (`\t`, `\n`, `,`, `;`, ,`<`, `>`).
4. Replace all found usernames within an email with `USERNAME@<domain>`.
5. Replace all found urls with `URL`.

In [8]:
# extract headers... slightly more convuluted

# firstly, the more complicated aspect is to figure out how to process one single line.
def filter_line_header(line: str):
    if len(line) == 0: return [], False
    # we will need to process a line character by character
    
    # check if line is continuation or new header (is tab or space present?)
    new_header = not (line[0] in {'\t', ' '})

    # single chars to skip over 
    single_char = {',', ';', '<', '>', '\t', '\n'}

    # url to keep track of 
    # we know we have a url when there is http present within the word
    http = ["h", "t", "t", "p"]
    http_idx = 0

    words = []
    curr = ""

    line += " " # ensure will run one last time
    for c in line:

        # , ; < > 
        if c in single_char:
            continue 

        # USERNAME@<domain>
        if c == "@": # replace username
            curr = "USERNAME@"
            continue
        
        # URL
        if http_idx == 3:
            pass
        elif c == http[http_idx]:
            http_idx += 1
        else:
            http_idx = 0

        # seperate
        if c == " ":
            
            if http_idx == 3: # meaning full http found.
                words.append("URL")
            elif len(curr) > 0: # take care of empty words (double spaces)
                words.append(curr)
            
            curr = ""
            escape = False
            http_idx = 0
            continue

        # add character if passes all these checks.
        curr += c
        
    return words, new_header

filter_line_header('exmh-workers-admin@spamassassin.taint.org \t\t\t\n \n\n\n\t\tt\<mailto:exmh-workers-request@spamassassin.taint.org?subject=help>;;;; <https://listman.spamassassin.taint.org/mailman/listinfo/exmh-workers>,     <<<mailto:exmh-workers-request@redhat.com?subject=unsubscribe>')

<>:60: SyntaxWarning: invalid escape sequence '\<'
<>:60: SyntaxWarning: invalid escape sequence '\<'
/var/folders/x4/j70qy1ns0gdg8282hfmvlbm80000gn/T/ipykernel_71195/2478648159.py:60: SyntaxWarning: invalid escape sequence '\<'
  filter_line_header('exmh-workers-admin@spamassassin.taint.org \t\t\t\n \n\n\n\t\tt\<mailto:exmh-workers-request@spamassassin.taint.org?subject=help>;;;; <https://listman.spamassassin.taint.org/mailman/listinfo/exmh-workers>,     <<<mailto:exmh-workers-request@redhat.com?subject=unsubscribe>')


(['USERNAME@spamassassin.taint.org',
  'USERNAME@spamassassin.taint.org?subject=help',
  'URL',
  'USERNAME@redhat.com?subject=unsubscribe'],
 True)

In [9]:
# now, to process all headers

# in previous iterations, we saved the index of the header to keep track of the order of the headers
# yet this isnt exactly necessary. so we have removed this feature.

# if it is needed again, added enumerate to our data and wrap our line lists in a tuple with each
# index.
def extract_headers(data: dict):
    '''
    Will transform a data object's `headers` key into a dictionary containing each header and its
    entries, numbered by order of appearance.
    '''

    d = dict()

    # sometimes lines continue, this is seen typically by a tab
    # we can also tell by determining that there is no header line present; is there <header name>:?

    curr_header = ''
    curr_val = []
    for line in data['headers']:
        l, new_header = filter_line_header(line)
        
        # process, if `:` present, it is a header title
        if new_header:
            # save previous values

            if d.get(curr_header):
                d[curr_header].append( curr_val )
            else:
                d[curr_header] = [ curr_val ]

            curr_header = l[0][:-1] # cut out `:`
            curr_val = l[1:]
        else: # means it is a continuation of the previous header
            curr_val += l

    # cleanup
    # save previous values
    if d.get(curr_header):
        d[curr_header].append( curr_val )
    else:
        d[curr_header] = [ curr_val ]

    return d

email1_processed_headers = extract_headers(email1_data)

email1_processed_headers

{'': [[]],
 'Return-Path': [['USERNAME@spamassassin.taint.org']],
 'Delivered-To': [['USERNAME@localhost.netnoteinc.com'],
  ['USERNAME@listman.spamassassin.taint.org']],
 'Received': [['from',
   'localhost',
   '(localhost',
   '[127.0.0.1])',
   'by',
   'phobos.labs.netnoteinc.com',
   '(Postfix)',
   'with',
   'ESMTP',
   'id',
   '5134943C34',
   'for',
   'USERNAME@localhost',
   'Wed',
   '21',
   'Aug',
   '2002',
   '11:18:36',
   '-0400',
   '(EDT)'],
  ['from',
   'phobos',
   '[127.0.0.1]',
   'by',
   'localhost',
   'with',
   'IMAP',
   '(fetchmail-5.9.0)',
   'for',
   'USERNAME@localhost',
   '(single-drop)',
   'Wed',
   '21',
   'Aug',
   '2002',
   '16:18:36',
   '+0100',
   '(IST)'],
  ['from',
   'listman.spamassassin.taint.org',
   '(listman.spamassassin.taint.org',
   '[66.187.233.211])',
   'by',
   'dogma.slashnull.org',
   '(8.11.6/8.11.6)',
   'with',
   'ESMTP',
   'id',
   'g7LFJuZ30902',
   'for',
   'USERNAME@jmason.org',
   'Wed',
   '21',
   'Aug',
 

Now, lets format our body. We can perform the following actions on the body:
1. Split all words into entries within a list.
2. Remove all html formatting. (Anything enclosed with `<tag >`). We will use beautiful soup for
this implementation. Although it may complicate our data processing, the benefit of having an already
tested and made library seems to outweigh the potential slowdown.
3. Strip all newlines, tabs, and non-alphanumeric characters. Allow (` `) only.
4. Replace all found usernames within an email with `USERNAME@<domain>`.
5. Replace all found urls with `URL`.
6. Leave all characters as uppercase.

In [10]:
from bs4 import BeautifulSoup
def extract_body(data: dict):
    # first we will combine all lines, then remove all html content. 
    # we must combine all lines within the body to ensure that there is no multi-line html cut off.

    content = ' '.join( data['body'])
    soup = BeautifulSoup(content, 'html.parser')
    text = soup.get_text(separator=' ', strip=True)

    # now, it will all be in one line...
    # allowed non alnum characters. `@` is included for email checks
    allowed = {' ', '@'} 

    # url to keep track of 
    # we know we have a url when there is http present within the word
    http = ["h", "t", "t", "p"]
    http_idx = 0

    words = []
    curr = ""

    text += " " # ensure will run one last time
    for c in text:

        # skip non alphanumeric characters
        if not (c in allowed or c.isalnum()):
            continue

        # USERNAME@<domain>
        if c == "@": # replace username
            curr = "USERNAME@"
            continue
        
        # URL
        if http_idx == 3:
            pass
        elif c == http[http_idx]:
            http_idx += 1
        else:
            http_idx = 0

        # seperate
        if c == " ":
            
            if http_idx == 3: # meaning full http found.
                words.append("URL")
            elif len(curr) > 0: # take care of empty words (double spaces)
                words.append(curr.upper())
            
            curr = ""
            http_idx = 0
            continue

        # add character if passes all these checks. (upper case)
        curr += c
        
    return words

email1_processed_body = extract_body(email1_data)

email1_processed_body

ModuleNotFoundError: No module named 'bs4'

Now lets build one processing function to extract all relevant data in one go.

In [ ]:
def process_email(email_path: str):
    filename = email_path.split('/')[-1]
    lines = []

    with open(email_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    unprocessed_email = extract_lines(lines)

    processed_headers = extract_headers(unprocessed_email)
    processed_body = extract_body(unprocessed_email)

    processed_email = {
        'name': filename,
        'frm' : unprocessed_email['frm'],
        'headers' : processed_headers,
        'body': processed_body
    }

    return processed_email

process_email('./data/easy_ham_2/00003.19be8acd739ad589cd00d8425bac7115')

{'name': '00003.19be8acd739ad589cd00d8425bac7115',
 'frm': ['From exmh-workers-admin@redhat.com  Wed Aug 21 16:18:36 2002\n'],
 'headers': {'': [[]],
  'Return-Path': [['USERNAME@spamassassin.taint.org']],
  'Delivered-To': [['USERNAME@localhost.netnoteinc.com'],
   ['USERNAME@listman.spamassassin.taint.org']],
  'Received': [['from',
    'localhost',
    '(localhost',
    '[127.0.0.1])',
    'by',
    'phobos.labs.netnoteinc.com',
    '(Postfix)',
    'with',
    'ESMTP',
    'id',
    '5134943C34',
    'for',
    'USERNAME@localhost',
    'Wed',
    '21',
    'Aug',
    '2002',
    '11:18:36',
    '-0400',
    '(EDT)'],
   ['from',
    'phobos',
    '[127.0.0.1]',
    'by',
    'localhost',
    'with',
    'IMAP',
    '(fetchmail-5.9.0)',
    'for',
    'USERNAME@localhost',
    '(single-drop)',
    'Wed',
    '21',
    'Aug',
    '2002',
    '16:18:36',
    '+0100',
    '(IST)'],
   ['from',
    'listman.spamassassin.taint.org',
    '(listman.spamassassin.taint.org',
    '[66.187.23

## Extract All Files
Lets now perform this function on all files. Starting with all files within the `easy_ham_2` folder.

In [ ]:
import os

def extract_dir(path, ls):
    with os.scandir(path) as entries:
        for entry in entries:
            print(entry.path, end=' ')
            processed_email = process_email(entry.path)
            ls.append(processed_email)


easy_ham_2 = []
extract_dir('./data/easy_ham_2', easy_ham_2)

easy_ham_2[-2]

./data/easy_ham_2/01128.efb36914ecb55d78a894591eff0843c5 ./data/easy_ham_2/00659.02e6dd777f837798533eae8f3b6a0491 ./data/easy_ham_2/00776.7df92458e9cf04b8873c406bde7d2fbe ./data/easy_ham_2/00116.409b29c26edef06268b4bfa03ef1367a ./data/easy_ham_2/00615.23556d88fcb1179b25083cfc41017f42 ./data/easy_ham_2/00715.c11e77af45a2debe41aed46b2be09d59 ./data/easy_ham_2/01200.7cad0240e6c9e66d013ca8a7c2268871 ./data/easy_ham_2/00834.c820e444255bc80fafd01933f05703d6 ./data/easy_ham_2/00568.4bfc868e8de382e58f6d9baee080b56a ./data/easy_ham_2/00439.4588d5306e105aa80c51978dfc505d3f ./data/easy_ham_2/00229.a13256f5a663bbfb8050a7abe6932558 ./data/easy_ham_2/01360.12e20ec23520b1c5515424bf77ed94c8 ./data/easy_ham_2/01165.6ac614b3ccada8b003ad8586c8b88e4e ./data/easy_ham_2/00297.cd323132d57e5cb2e8599f69f6ff2848 ./data/easy_ham_2/00950.faa69102e30c15ae98cdfee27a2763a7 ./data/easy_ham_2/01346.aa3fbf33029311844175e5d6a87aece9 ./data/easy_ham_2/00955.65d74e4da04399281b0b4af1e8e17734 ./data/easy_ham_2/00999.8251f0e

{'name': '01178.5c977dff972cd6eef64d4173b90307f0',
 'frm': ['From rpm-list-admin@freshrpms.net  Tue Jul 30 22:25:48 2002\n'],
 'headers': {'': [[]],
  'Return-Path': [['USERNAME@freshrpms.net']],
  'Delivered-To': [['USERNAME@localhost.netnoteinc.com']],
  'Received': [['from',
    'localhost',
    '(localhost',
    '[127.0.0.1])',
    'by',
    'phobos.labs.netnoteinc.com',
    '(Postfix)',
    'with',
    'ESMTP',
    'id',
    'CB18D4407C',
    'for',
    'USERNAME@localhost',
    'Tue',
    '30',
    'Jul',
    '2002',
    '17:25:47',
    '-0400',
    '(EDT)'],
   ['from',
    'phobos',
    '[127.0.0.1]',
    'by',
    'localhost',
    'with',
    'IMAP',
    '(fetchmail-5.9.0)',
    'for',
    'USERNAME@localhost',
    '(single-drop)',
    'Tue',
    '30',
    'Jul',
    '2002',
    '22:25:47',
    '+0100',
    '(IST)'],
   ['from',
    'egwn.net',
    '(ns2.egwn.net',
    '[193.172.5.4])',
    'by',
    'dogma.slashnull.org',
    '(8.11.6/8.11.6)',
    'with',
    'ESMTP',
    'i

In [ ]:
hard_ham = []
extract_dir('./data/hard_ham', hard_ham)

print() # newline
spam_2 = []
extract_dir('./data/spam_2', spam_2)

./data/hard_ham/00113.1d37bdbcad4975b5012cc6d87a048ecf ./data/hard_ham/00165.4e1923f1c9091dc00d79dfe004eedf26 ./data/hard_ham/00044.b2f03a4d512deb2c4702a3bd60a4fd88 ./data/hard_ham/00138.1bf66894faa2d923b2a673a016dc6afd ./data/hard_ham/00211.1ca67c3677fa0051ccfa27c756d32b5a ./data/hard_ham/00201.04e4c8ef93080eea4b11213262ece700 ./data/hard_ham/00013.15135df1ed8198dbea3fcd0cb8d071ae ./data/hard_ham/00019.e35a7a6a1a6bdd0d2e164db2f6a0e4ef ./data/hard_ham/00072.7fdfa76485745886daefc536dc734132 ./data/hard_ham/00223.14b06feeb8b03fed4e272140b8ed95f0 ./data/hard_ham/00074.c6f6a8cef7af2b318f769fc4633aa68d ./data/hard_ham/00151.b352916ecff2b0ba1140d6898d789235 ./data/hard_ham/00102.10533c2a8486368eea6acd0d07549f3b ./data/hard_ham/00073.4bd106f62a13d9071a638228a7189862 ./data/hard_ham/00014.a1f7ca2723b9e4060e7c73b6e1fed642 ./data/hard_ham/00080.338dc467b34650c3fb2deebfb49c7279 ./data/hard_ham/00166.3f2f67be8df73f6566634579b2a4a5a6 ./data/hard_ham/00197.c7483488867fe74e7444441283549ae8 ./data/har

## Exploratory Data Anaylsis
Continuing forward, we still have much preprocessing and data analysis to complete before we move 
forward. Firstly, lets find the most common words found in bodies.

In [ ]:
all_ham = easy_ham_2 + hard_ham

In [ ]:
from email_processor import EmailProcessorV1